# D222 - pandas with HDFS

Create sample e-commerce order data with pandas, write it to HDFS as CSV, read it back into pandas, convert it to JSON Lines, and verify both files with HDFS commands.

> Run this notebook from Jupyter inside Ubuntu/WSL. HDFS must be running.

## 1. Driver and notebook setup

This lab deliberately uses the installed Hadoop command-line client as the HDFS transport. Python sends data to and receives data from `hdfs dfs` through standard input/output.

**No additional HDFS driver is required.** In particular, this approach does not require `libhdfs`, PyArrow's Hadoop bindings, WebHDFS, or an extra HDFS Python package. It requires only:

- Java 11 and Hadoop 3.3.6 already configured
- a running HDFS NameNode and DataNode
- Python, pandas, JupyterLab, and an IPython kernel

One-time Ubuntu/WSL setup, if pandas and Jupyter are not already installed:

```bash
sudo apt update
sudo apt install -y python3-venv
python3 -m venv ~/venvs/dataeng
source ~/venvs/dataeng/bin/activate
python -m pip install --upgrade pip
python -m pip install pandas jupyterlab ipykernel
python -m ipykernel install --user --name dataeng --display-name 'Python (dataeng)'
jupyter lab --no-browser
```

Choose the **Python (dataeng)** kernel. Start Jupyter from a shell where `JAVA_HOME`, `HADOOP_HOME`, `HADOOP_CONF_DIR`, and the Hadoop `bin` directory are available.

## 2. Check the WSL, Python, and Hadoop environment

The notebook kernel inherits environment variables and `PATH` from the process that started Jupyter. If `hdfs` is not found here, stop Jupyter, source `~/.bashrc`, and start it again.

In [ ]:
import os
import platform
import shutil
import sys

import pandas as pd

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Platform:", platform.platform())
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "not set"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME", "not set"))
print("hdfs command:", shutil.which("hdfs") or "not found")

if shutil.which("hdfs") is None:
    raise RuntimeError(
        "The hdfs command is not on PATH. Start Jupyter after sourcing ~/.bashrc."
    )

Confirm that HDFS is reachable and has a live DataNode. A directory operation can use only the NameNode; writing the sample later also tests the DataNode.

In [ ]:
%%bash
echo "HDFS endpoint: $(hdfs getconf -confKey fs.defaultFS)"
hdfs dfsadmin -safemode get
hdfs dfsadmin -report | grep -E 'Live datanodes|Dead datanodes'

## 3. Create sample e-commerce orders

Dates, quantities, and prices use explicit pandas data types. `order_total` is calculated rather than typed manually.

In [ ]:
orders = pd.DataFrame(
    {
        "order_id": [1001, 1002, 1003, 1004, 1005, 1006],
        "order_date": pd.to_datetime(
            ["2026-08-10", "2026-08-10", "2026-08-11",
             "2026-08-12", "2026-08-12", "2026-08-13"]
        ),
        "customer_id": ["C101", "C102", "C101", "C103", "C104", "C102"],
        "product": ["Keyboard", "Monitor", "Mouse", "USB Hub", "Laptop Stand", "Webcam"],
        "category": ["Accessories", "Displays", "Accessories",
                     "Accessories", "Furniture", "Video"],
        "quantity": pd.Series([1, 2, 3, 1, 1, 2], dtype="int64"),
        "unit_price": [2499.00, 12999.50, 799.00, 1499.00, 2899.00, 3499.00],
        "status": ["SHIPPED", "PROCESSING", "DELIVERED",
                   "CANCELLED", "SHIPPED", "PROCESSING"],
    }
)
orders["order_total"] = (orders["quantity"] * orders["unit_price"]).round(2)
orders

Inspect the schema and a few basic totals before writing. This catches data-type mistakes early.

In [ ]:
orders.info()

display(
    orders.groupby("status", as_index=False)
    .agg(order_count=("order_id", "count"), revenue=("order_total", "sum"))
)

## 4. Create small HDFS helper functions

`subprocess.run` executes Hadoop directly without invoking a shell. This avoids shell quoting problems. Failures raise an exception containing Hadoop's error message.

In [ ]:
from io import StringIO
import subprocess
from typing import Sequence


def run_hdfs(arguments: Sequence[str], *, input_text: str | None = None) -> str:
    """Run an `hdfs dfs` command and return its standard output."""
    command = ["hdfs", "dfs", *arguments]
    completed = subprocess.run(
        command,
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.returncode != 0:
        message = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(f"HDFS command failed: {' '.join(command)}\n{message}")
    return completed.stdout


def hdfs_write_text(path: str, content: str, *, overwrite: bool = True) -> None:
    """Stream text to an HDFS file without creating a local temporary file."""
    arguments = ["-put"]
    if overwrite:
        arguments.append("-f")
    arguments.extend(["-", path])
    run_hdfs(arguments, input_text=content)


def hdfs_read_text(path: str) -> str:
    """Stream an HDFS text file into Python."""
    return run_hdfs(["-cat", path])

## 5. Write the pandas DataFrame to HDFS as CSV

The requested relative path `demo/orders/orders.csv` resolves to `/user/$USER/demo/orders/orders.csv`. We use the absolute path so the destination is unambiguous. `index=False` prevents pandas from adding its row index as an extra CSV column.

In [ ]:
hdfs_home = run_hdfs(["-getHomeDirectory"]).strip()
hdfs_orders_dir = f"{hdfs_home}/demo/orders"
hdfs_csv_path = f"{hdfs_orders_dir}/orders.csv"
hdfs_json_path = f"{hdfs_orders_dir}/orders.json"

run_hdfs(["-mkdir", "-p", hdfs_orders_dir])
csv_text = orders.to_csv(index=False, date_format="%Y-%m-%d")
hdfs_write_text(hdfs_csv_path, csv_text)

print("Wrote:", hdfs_csv_path)
print(run_hdfs(["-ls", "-h", hdfs_orders_dir]))

During this write, pandas formats the DataFrame in Python. The HDFS client contacts the NameNode to create the path and obtain a block target, then streams CSV bytes to the DataNode.

## 6. Verify the CSV with HDFS commands

These cells use the HDFS shell directly, independently of the Python helper.

In [ ]:
%%bash
ORDERS_DIR="/user/$USER/demo/orders"
hdfs dfs -ls -h "$ORDERS_DIR"
echo
hdfs dfs -stat 'name=%n bytes=%b block_size=%o replication=%r owner=%u mode=%a' "$ORDERS_DIR/orders.csv"

In [ ]:
%%bash
echo '=== CSV content from HDFS ==='
hdfs dfs -cat "/user/$USER/demo/orders/orders.csv"

## 7. Read the HDFS CSV into pandas

`hdfs_read_text` streams the complete file from a DataNode. `StringIO` gives pandas a file-like text object. Explicit date parsing restores `order_date` as a pandas datetime column. This pattern is suitable for instructional and modest-sized files; pandas still loads the full dataset into one machine's memory.

In [ ]:
csv_from_hdfs = hdfs_read_text(hdfs_csv_path)
orders_from_hdfs = pd.read_csv(
    StringIO(csv_from_hdfs),
    parse_dates=["order_date"],
)

orders_from_hdfs

Validate that the round trip preserved row count, column names, order IDs, and calculated totals.

In [ ]:
assert len(orders_from_hdfs) == len(orders)
assert orders_from_hdfs.columns.tolist() == orders.columns.tolist()
assert orders_from_hdfs["order_id"].tolist() == orders["order_id"].tolist()
pd.testing.assert_series_equal(
    orders_from_hdfs["order_total"],
    orders["order_total"],
    check_names=False,
)
print(f"Validated {len(orders_from_hdfs)} rows from HDFS.")

## 8. Transform and write JSON Lines to HDFS

JSON Lines stores one JSON object per line. It is easier for distributed tools to split and stream than a single JSON array. `date_format='iso'` writes dates in an interoperable ISO representation.

In [ ]:
json_text = orders_from_hdfs.to_json(
    orient="records",
    lines=True,
    date_format="iso",
)
hdfs_write_text(hdfs_json_path, json_text)

print("Wrote:", hdfs_json_path)
print(run_hdfs(["-ls", "-h", hdfs_orders_dir]))

## 9. List and display the JSON using HDFS commands

In [ ]:
%%bash
ORDERS_DIR="/user/$USER/demo/orders"
echo '=== HDFS directory ==='
hdfs dfs -ls -h "$ORDERS_DIR"
echo
echo '=== JSON Lines content from HDFS ==='
hdfs dfs -cat "$ORDERS_DIR/orders.json"

Read the JSON from HDFS back into pandas as a second round-trip check.

In [ ]:
json_from_hdfs = hdfs_read_text(hdfs_json_path)
orders_from_json = pd.read_json(
    StringIO(json_from_hdfs),
    orient="records",
    lines=True,
)
orders_from_json

## 10. Download files with `hdfs dfs -get`

`-cat` streams content to the terminal. `-get` creates a local Linux file. The following downloads into `/tmp/d222-download`, not into HDFS.

In [ ]:
%%bash
DOWNLOAD_DIR=/tmp/d222-download
mkdir -p "$DOWNLOAD_DIR"
hdfs dfs -get -f "/user/$USER/demo/orders/orders.csv" "$DOWNLOAD_DIR/orders.csv"
hdfs dfs -get -f "/user/$USER/demo/orders/orders.json" "$DOWNLOAD_DIR/orders.json"
ls -lh "$DOWNLOAD_DIR"
echo
head -3 "$DOWNLOAD_DIR/orders.csv"
head -2 "$DOWNLOAD_DIR/orders.json"

## 11. Important limitation

pandas is a single-machine library. HDFS can store very large files, but this notebook's helper returns an entire file as one Python string and pandas loads it into WSL memory. For data larger than available memory, use a distributed engine such as Spark, Hive, or a chunked/partitioned workflow.

Direct URLs such as `pd.read_csv('hdfs://...')` require an additional filesystem implementation and native/runtime configuration. That adds failure points without improving this introductory lab, so the Hadoop CLI bridge is used here.

## 12. Optional cleanup

Run only when the demonstration files are no longer needed. It removes the HDFS demo directory and the local download directory.

In [ ]:
%%bash
HDFS_DEMO="/user/$USER/demo/orders"
case "$HDFS_DEMO" in
  /user/*/demo/orders) hdfs dfs -rm -r -f "$HDFS_DEMO" ;;
  *) echo "Refusing unexpected cleanup path: $HDFS_DEMO"; exit 1 ;;
esac
rm -rf /tmp/d222-download
echo 'D222 demonstration files removed.'

## Workflow recap

```text
pandas DataFrame
    -> DataFrame.to_csv()
    -> hdfs dfs -put - /user/$USER/demo/orders/orders.csv
    -> hdfs dfs -cat
    -> pandas.read_csv(StringIO(...))
    -> DataFrame.to_json(lines=True)
    -> hdfs dfs -put - /user/$USER/demo/orders/orders.json
    -> hdfs dfs -ls / -cat / -get for verification
```